In [7]:
import numpy as np
import pandas as pd
import joblib
from itertools import product
from functools import partial
from multiprocessing import Pool
from tqdm import tqdm
from physics import *

In [8]:
def process_combination(params, n_layers=1):
    """
    params: [k0, k1, ..., k_{n+1}, h1, h2, ..., hn]
    """
    r_val = forward_model(params=params[:-1], theta=params[-1], n_layers=n_layers)

    result_dict = {
        'theta': params[-1],
        'h': params[n_layers+2:-1],
        'k_up': params[0],
        'k_down': params[n_layers+1],
        'k_vals': params[1:n_layers+1],
        'Re(r)': np.real(r_val),
        'Im(r)': np.imag(r_val)
    }
    return result_dict

def generation_params(k_range, h_range, theta_range, n_layers=1):
    thetas = np.deg2rad(np.arange(theta_range[0], theta_range[1], 10))
    hs = np.linspace(h_range[0], h_range[1], 10)
    k_vals = np.linspace(k_range[0], k_range[1], 10)

    params_iter = product([1], *[k_vals]*n_layers, [1], *[hs]*n_layers, thetas)
    expected_size = len(k_vals) ** n_layers * len(hs) ** n_layers * len(thetas)
    return params_iter, expected_size

In [9]:
n_layers = 2
chunksize = 50

In [10]:
params_iter, expected_size = generation_params(k_range=[0.1,1], h_range=[0.1, 3], theta_range=[10, 90], n_layers=n_layers)

worker = partial(process_combination, n_layers=n_layers)

with Pool() as pool:
    results = list(
        tqdm(
            pool.imap_unordered(worker, params_iter, chunksize=chunksize),
            total=expected_size
        )
    )

100%|██████████| 80000/80000 [00:09<00:00, 8830.18it/s] 


In [11]:
data = pd.DataFrame(results)

display(data, data.info())

<class 'pandas.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   theta   80000 non-null  float64
 1   h       80000 non-null  object 
 2   k_up    80000 non-null  int64  
 3   k_down  80000 non-null  int64  
 4   k_vals  80000 non-null  object 
 5   Re(r)   80000 non-null  float64
 6   Im(r)   80000 non-null  float64
dtypes: float64(3), int64(2), object(2)
memory usage: 4.3+ MB


,theta,h,k_up,k_down,k_vals,Re(r),Im(r)
0,0.872665,"(0.42222222222222217, 0.7444444444444444)",1,1,"(0.1, 0.1)",-0.084343,-0.709314
1,1.047198,"(0.42222222222222217, 0.7444444444444444)",1,1,"(0.1, 0.1)",-0.321075,-0.738656
2,1.221730,"(0.42222222222222217, 0.7444444444444444)",1,1,"(0.1, 0.1)",-0.616117,-0.653584
3,1.396263,"(0.42222222222222217, 0.7444444444444444)",1,1,"(0.1, 0.1)",-0.886322,-0.397805
4,0.174533,"(0.42222222222222217, 1.0666666666666667)",1,1,"(0.1, 0.1)",0.347671,-0.491479
...,...,...,...,...,...,...,...
79995,0.698132,"(3.0, 3.0)",1,1,"(1.0, 1.0)",0.000000,-0.000000
79996,0.872665,"(3.0, 3.0)",1,1,"(1.0, 1.0)",-0.000000,-0.000000
79997,1.047198,"(3.0, 3.0)",1,1,"(1.0, 1.0)",-0.000000,-0.000000
79998,1.221730,"(3.0, 3.0)",1,1,"(1.0, 1.0)",-0.000000,-0.000000


None

In [12]:
# joblib.dump(data, 'data/one_slay.joblib')